In [7]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import duckdb

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

## 1. Load & Clean with DuckDB

**Optimization:** Using **DuckDB** for out-of-core SQL processing. This allows us to query and clean the Parquet file directly from disk without loading it all into RAM, preventing kernel crashes.

In [8]:
# Define paths
input_path = Path('../data/processed/integrated_raw.parquet')
output_path = Path('../data/processed/integrated_clean.parquet')

if not input_path.exists():
    raise FileNotFoundError(
        "Integrated dataset not found! Please run notebook 04_integrate_datasets.ipynb first."
    )

print("Initializing DuckDB connection...")
con = duckdb.connect(database=':memory:')

# Inspect schema to build dynamic query
print("Inspecting schema...")
columns_info = con.execute(f"DESCRIBE SELECT * FROM '{input_path}'").fetchall()
columns = [col[0] for col in columns_info]
print(f"✓ Found {len(columns)} columns")

Initializing DuckDB connection...
Inspecting schema...
✓ Found 25 columns


## 2. Construct Cleaning Query

We dynamically build a SQL query to:
1. Select columns (handling renames and drops)
2. Deduplicate rows (`DISTINCT`)

In [9]:
print("Constructing SQL query...")

# Identify redundant columns
crash_suffix_cols = [c for c in columns if c.endswith('_crash')]

select_clauses = []
dropped_cols = []
renamed_cols = []

# Track processed columns to avoid duplicates in selection
processed_cols = set()

# Handle suffix columns first
for crash_col in crash_suffix_cols:
    base_name = crash_col.replace('_crash', '')
    person_col = base_name + '_person'
    
    if person_col in columns:
        # We have both. In SQL we'll select the crash one as the base name
        select_clauses.append(f'"{crash_col}" AS "{base_name}"')
        renamed_cols.append(base_name)
        dropped_cols.append(person_col)
        
        processed_cols.add(crash_col)
        processed_cols.add(person_col)

# Add remaining columns
for col in columns:
    if col not in processed_cols:
        select_clauses.append(f'"{col}"')

query = f"""
    SELECT DISTINCT
        {','.join(select_clauses)}
    FROM '{input_path}'
"""

print(f"✓ Plan created:")
print(f"  - Dropping {len(dropped_cols)} redundant columns")
print(f"  - Renaming {len(renamed_cols)} columns")
print(f"  - Deduplicating rows (DISTINCT)")

Constructing SQL query...
✓ Plan created:
  - Dropping 0 redundant columns
  - Renaming 0 columns
  - Deduplicating rows (DISTINCT)


## 3. Execute and Save

Use DuckDB's `COPY` command to stream the result of the query directly to a Parquet file.

In [ ]:
print("Executing pipeline and saving to disk...")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Execute COPY command
copy_query = f"COPY ({query}) TO '{output_path}' (FORMAT PARQUET, COMPRESSION 'SNAPPY')"
con.execute(copy_query)

print("✅ CLEANED INTEGRATED DATASET SAVED!")
print("="*60)
print(f"Path: {output_path}")
print(f"Size: {output_path.stat().st_size / (1024**2):.2f} MB")

# Verify
count = con.execute(f"SELECT COUNT(*) FROM '{output_path}'").fetchone()[0]
print(f"Records: {count:,}")

con.close()

Executing pipeline and saving to disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ CLEANED INTEGRATED DATASET SAVED!
Path: ..\data\processed\integrated_clean.parquet
Size: 134.60 MB
Records: 5,385,654


: 